# 05 — Final Load Preparation

**Goal:** Transform the cleaned dataset into a high-fidelity **Tableau-ready master file**. This includes calculating micro-market benchmarks and ensuring field names are optimized for BI visualization.

## 5.1 Environment Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == 'notebooks'
    else Path.cwd().resolve()
)
INPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'airbnb_nyc_cleaned.csv'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'nyc_airbnb_tableau_master.csv'

df = pd.read_csv(INPUT_PATH)
print(f"Loaded {df.shape[0]:,} listings for final preparation.")

## 5.2 Micro-Market Benchmarking (Price Index)

**Logic:** We calculate the median price for every neighborhood. A listing's **Price Index** is its price divided by this median. 
- Value > 1.0: Premium pricing strategy
- Value < 1.0: Value pricing strategy

In [ ]:
# Calculate Neighborhood Medians
hood_medians = df.groupby(['neighbourhood_group', 'neighbourhood'])['price'].transform('median')

# Compute Price Index
df['price_index'] = (df['price'] / hood_medians).round(2)

print("Price Index calculated successfully.")

## 5.3 Listing Demand Score & Yield

**Demand Score:** `reviews_per_month * (availability_365 / 365)`  
**Yield Potential:** `revenue_proxy / price`

In [ ]:
# Demand Score (Listing Demand Score)
df['demand_score'] = (df['review_rate_month'] * (df['availability_365'] / 365.0)).round(4)

# Yield Potential (Efficiency check)
df['yield_potential'] = np.where(df['price'] > 0, (df['revenue_proxy'] / df['price']), 0).astype(float)

print("Efficiency KPIs generated.")

## 5.4 Activity & Seasonality Filters

Preparing categorical dimensions for Tableau filters.

In [ ]:
# Activity Flag
def categorize_activity(year):
    if pd.isna(year): return 'No Reviews'
    if year == 2019: return 'Active (2019)'
    if year == 2018: return 'Trailing (2018)'
    return 'Dormant (<2018)'

df['activity_status'] = df['review_year'].apply(categorize_activity)

# Seasonality Peak (Summer = June, July, August)
df['is_summer_peak'] = df['review_month'].isin([6, 7, 8]).map({True: 'Summer Peak', False: 'Off-Peak'})

print("Categorical dimensions updated.")

## 5.5 Professional Formatting & Export

Renaming columns to match the **Data Dictionary** for a seamless Tableau import.

In [ ]:
column_map = {
    'id': 'Listing ID',
    'name': 'Listing Title',
    'host_id': 'Host ID',
    'host_name': 'Host Name',
    'neighbourhood_group': 'Borough',
    'neighbourhood': 'Neighborhood',
    'latitude': 'Latitude',
    'longitude': 'Longitude',
    'room_type': 'Room Type',
    'price': 'Price',
    'minimum_nights': 'Min Nights',
    'review_count': 'Total Reviews',
    'last_review': 'Last Review Date',
    'review_rate_month': 'Reviews per Month',
    'host_listing_count': 'Host Portfolio Size',
    'availability_365': 'Availability 365',
    'revenue_proxy': 'Annual Revenue Proxy',
    'price_index': 'Market Price Index',
    'demand_score': 'Listing Demand Score',
    'yield_potential': 'Yield Potential',
    'activity_status': 'Recent Activity Status',
    'is_summer_peak': 'Seasonality Status',
    'host_portfolio_segment': 'Host Segment',
    'price_tier': 'Price Category'
}

df_tableau = df.rename(columns=column_map)

# Select final columns defined in dictionary
final_cols = [c for c in column_map.values() if c in df_tableau.columns]
df_tableau = df_tableau[final_cols]

df_tableau.to_csv(OUTPUT_PATH, index=False)
print(f"Success! Tableau Master Dataset exported to: {OUTPUT_PATH}")